# MCP Protocol Messages
An MCP session is a sequence of JSON-RPC 2.0 messages exchanged between a client and a server. This lecture takes a session apart message by message: the three message shapes, the fixed lifecycle they follow, tool discovery and invocation, and the stdio transport that carries them. Every quoted message below was captured from this module's server during an actual session; the demo notebook shows the same messages crossing the wire live.

> __Learning Objectives__
>
> By the end of this lecture, you will be able to:
> * __Read the three message shapes:__ Construct and read JSON-RPC 2.0 requests, responses, and notifications, and interpret the error object and its standard codes.
> * __Trace the session lifecycle:__ Follow an MCP session from subprocess launch through the initialize handshake and tool operations to shutdown at end-of-file.
> * __Distinguish the two failure modes:__ Tell a protocol error (a JSON-RPC error object) from a tool execution error (a result with `isError` set to `true`), and explain why hosts treat them differently.

To read the captured exchanges, we first need the vocabulary they are written in. Let's get started!
___

## JSON-RPC 2.0 Message Anatomy
MCP builds on JSON-RPC 2.0, a remote procedure call format in which every message is a small JSON object. Three message shapes cover everything the protocol needs:

> __Request__
>
> A message that asks the other side to perform an operation and expects an answer. A request carries three required fields, `jsonrpc` (always the string `"2.0"`), `id` (a number that pairs the request with its response), and `method` (the operation name, e.g., `tools/list`), plus an optional `params` object of arguments, present when the method takes any.

> __Response__
>
> The answer to a request, carrying the same `id` as the request it answers. A response holds exactly one of two fields: `result` when the operation succeeded, or `error` when the request failed. Never both.

> __Notification__
>
> A one-way message with a `method` but no `id`. A notification is never answered; the receiver processes it and sends nothing back.

When a request fails, the `error` field is an object with a numeric `code` and a human-readable `message`. This module's server uses the standard JSON-RPC codes:

| Code | Meaning |
| :-- | :-- |
| `-32700` | Parse error: the line was not valid JSON. |
| `-32601` | Method not found: the server does not implement the requested `method`. |
| `-32602` | Invalid params: the parameters are invalid, e.g., an unknown tool name. |
| `-32603` | Internal error: the server failed while handling the request. |

These three shapes are the complete vocabulary of a session. Next, we trace the order in which they appear.
___

## Session Lifecycle
An MCP session over stdio has a fixed opening and closing, with tool operations in between.

### Algorithm
The session proceeds in five steps:

1. __Launch:__ The host starts the server as a subprocess and connects to its standard input and output.
2. __Initialize:__ The client sends an `initialize` request carrying its `protocolVersion`, `capabilities`, and `clientInfo`. The server answers with its own protocol version, its capabilities, and a `serverInfo` block naming the server.
3. __Confirm:__ The client sends the `notifications/initialized` notification, which has no `id` and receives no response.
4. __Operate:__ The client issues `tools/list` and `tools/call` requests in any order, matching responses to requests by `id`.
5. __Shutdown:__ The client closes the server's standard input; the server exits when its input reaches end-of-file (EOF).

Here is the captured initialize handshake from this module's server. Messages labeled `Client → Server` travel down the subprocess's standard input; messages labeled `Server → Client` come back on its standard output.

**Client → Server**

```json
{
  "method": "initialize",
  "id": 1,
  "params": {
    "clientInfo": {
      "name": "cheme-142-notebook-client",
      "version": "1.0.0"
    },
    "protocolVersion": "2025-06-18",
    "capabilities": {}
  },
  "jsonrpc": "2.0"
}
```

**Server → Client**

```json
{
  "id": 1,
  "jsonrpc": "2.0",
  "result": {
    "protocolVersion": "2025-06-18",
    "capabilities": {
      "tools": {}
    },
    "serverInfo": {
      "name": "cheme-142-m4-cheme-tools",
      "version": "1.0.0"
    }
  }
}
```

**Client → Server**

```json
{
  "method": "notifications/initialized",
  "jsonrpc": "2.0"
}
```

Note the third message: it has a `method` but no `id`, so it is a notification. The server sends nothing back, and the session moves on to operation.
___

## Tool Discovery
After the handshake, the client learns what the server can do. The `tools/list` request takes no parameters, and the result carries one descriptor per tool. Each descriptor has three fields: `name`, the identifier used to invoke the tool; `description`, human-readable text a host passes to its model; and `inputSchema`, a JSON Schema object whose `type`, `properties`, and `required` fields specify the arguments the tool accepts.

**Client → Server**

```json
{
  "method": "tools/list",
  "id": 2,
  "params": {},
  "jsonrpc": "2.0"
}
```

**Server → Client**

```json
{
  "id": 2,
  "jsonrpc": "2.0",
  "result": {
    "tools": [
      {
        "name": "antoine_vapor_pressure",
        "inputSchema": {
          "properties": {
            "T": {
              "type": "number",
              "description": "Temperature in K, inside the species' valid range"
            },
            "species": {
              "type": "string",
              "description": "Species name, e.g. water, acetone, ethanol, benzene"
            }
          },
          "required": [
            "species",
            "T"
          ],
          "type": "object"
        },
        "description": "Saturation pressure (bar) of a named species at temperature T (K) from the Antoine equation."
      },
      {
        "name": "ideal_gas_solve",
        "inputSchema": {
          "properties": {
            "T": {
              "type": "number",
              "description": "Temperature in K"
            },
            "P": {
              "type": "number",
              "description": "Pressure in Pa"
            },
            "V": {
              "type": "number",
              "description": "Volume in m^3"
            },
            "n": {
              "type": "number",
              "description": "Amount in mol"
            }
          },
          "required": [],
          "type": "object"
        },
        "description": "Solve the ideal gas law PV = nRT for the one variable not provided (SI units: Pa, m^3, mol, K)."
      },
      {
        "name": "molecular_weight",
        "inputSchema": {
          "properties": {
            "formula": {
              "type": "string",
              "description": "Chemical formula, element symbols with optional integer counts"
            }
          },
          "required": [
            "formula"
          ],
          "type": "object"
        },
        "description": "Compute the molar mass (g/mol) of a chemical formula, e.g. H2O or C6H12O6."
      }
    ]
  }
}
```

The `inputSchema` is the contract an agent uses to construct arguments. For `antoine_vapor_pressure`, the schema names two properties, a string `species` and a number `T`, and requires both. A caller needs nothing beyond this descriptor: the schema is the tool's complete interface.
___

## Tool Invocation
To invoke a tool, the client sends a `tools/call` request whose `params` carry two entries: the tool `name` and an `arguments` object that conforms to the tool's input schema. The result carries a `content` array of typed items (text items in this module) and an `isError` flag, which is `false` on success. Here is the captured call to `molecular_weight`:

**Client → Server**

```json
{
  "method": "tools/call",
  "id": 3,
  "params": {
    "name": "molecular_weight",
    "arguments": {
      "formula": "C6H12O6"
    }
  },
  "jsonrpc": "2.0"
}
```

**Server → Client**

```json
{
  "id": 3,
  "jsonrpc": "2.0",
  "result": {
    "content": [
      {
        "text": "{\"units\":\"g/mol\",\"formula\":\"C6H12O6\",\"molar_mass\":180.156}",
        "type": "text"
      }
    ],
    "isError": false
  }
}
```

The text item's payload is itself a JSON string produced by the tool; the client parses it to recover the values.

Not every call succeeds, and the protocol separates two kinds of failure. First, a request that names a tool the server does not have:

**Client → Server**

```json
{
  "method": "tools/call",
  "id": 5,
  "params": {
    "name": "gibbs_energy",
    "arguments": {
      "species": "water"
    }
  },
  "jsonrpc": "2.0"
}
```

**Server → Client**

```json
{
  "error": {
    "message": "Unknown tool: gibbs_energy",
    "code": -32602
  },
  "id": 5,
  "jsonrpc": "2.0"
}
```

Second, a valid request whose tool runs and fails, here a temperature outside the species' valid range:

**Client → Server**

```json
{
  "method": "tools/call",
  "id": 6,
  "params": {
    "name": "antoine_vapor_pressure",
    "arguments": {
      "T": 500.0,
      "species": "water"
    }
  },
  "jsonrpc": "2.0"
}
```

**Server → Client**

```json
{
  "id": 6,
  "jsonrpc": "2.0",
  "result": {
    "content": [
      {
        "text": "ArgumentError: T = 500.0 K is outside the valid range [255.9, 373.0] K for water",
        "type": "text"
      }
    ],
    "isError": true
  }
}
```

Compare the two responses:

> __Protocol error:__ The request itself was bad. The server returns a JSON-RPC error object (code `-32602` here) and no `result`.

> __Tool execution error:__ The request was valid, but the tool failed. The server returns a normal `result` with `isError` set to `true` and the failure text in `content`.

Hosts handle these differently: a protocol error signals a malformed request, while a tool execution error is a legitimate result the host reports back to the model.
___

## The stdio Transport
Every message above traveled the same way: as one line of text between two processes. The stdio transport frames messages as newline-delimited UTF-8: one complete JSON-RPC message per line, with no embedded newlines inside a message. The server's standard output is reserved for protocol messages; logs and any other output go to standard error, so they never corrupt the message stream.

The server's side of the transport is a dispatch loop.

### Algorithm
The server dispatch loop processes one message per iteration:

1. __Read:__ Read one line from standard input.
2. __Parse:__ Parse the line as a JSON-RPC message; on failure, emit a `-32700` parse error.
3. __Route:__ Route the message by its `method` to the matching handler; notifications produce no response.
4. __Respond:__ Write the response as a single line to standard output and flush.
5. __Repeat:__ Continue until standard input reaches EOF, then exit.

This loop is the entire runtime of this module's server; `src/Server.jl` implements it directly, and closing the server's standard input is what ends a session.
___

## Summary
This lecture took an MCP session apart message by message: the three JSON-RPC 2.0 shapes, the lifecycle they follow, the discovery and invocation exchanges, and the stdio framing underneath.

> __Key Takeaways:__
>
> * **Three message shapes:** Every MCP message is a request (carries an `id` and expects an answer), a response (`result` or `error`, matched by `id`), or a notification (no `id`, never answered).
> * **A fixed lifecycle:** A session runs initialize → initialized → operate → EOF: the handshake opens it, `tools/list` and `tools/call` do the work, and closing the server's standard input ends it.
> * **Two error channels:** A protocol error returns a JSON-RPC error object with no `result`; a tool execution error returns a normal `result` with `isError` set to `true`. Hosts treat the first as a malformed request and the second as a result to report.

The demo notebook drives every exchange in this lecture against the live server, and the activities have you construct the same messages yourself.
___